In [1]:
!pip install xgboost

In [2]:
import os
os.chdir("..")
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier

In [3]:
FEATURE_COLS = ["hr", "hrv", "eda", "wrist_temp", "co2_noisy", "lux_noisy", "posture_cm"]
df = pd.read_csv("outputs/unified_features.csv").dropna(subset=FEATURE_COLS)

le = LabelEncoder()
y = le.fit_transform(df["ground_truth"])
groups = df["subject"].values
X = df[FEATURE_COLS].values

scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.3f}")

scale_pos_weight = 2.337


In [4]:
from sklearn.model_selection import StratifiedGroupKFold

N_REPEATS = 10
fold_f1 = []
all_preds, all_true = [], []

for repeat in range(N_REPEATS):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
    for train_idx, test_idx in sgkf.split(X, y, groups=groups):
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[train_idx])
        Xte = scaler.transform(X[test_idx])
        clf = XGBClassifier(n_estimators=200, max_depth=4, scale_pos_weight=scale_pos_weight,
                             eval_metric="logloss", random_state=42)
        clf.fit(Xtr, y[train_idx])
        preds = clf.predict(Xte)
        all_preds.extend(preds)
        all_true.extend(y[test_idx])
        fold_f1.append(f1_score(y[test_idx], preds, average="macro", zero_division=0))

print(f"XGBoost: mean macro-F1 = {np.mean(fold_f1):.4f} (SD={np.std(fold_f1):.4f}), n={len(fold_f1)}")

XGBoost: mean macro-F1 = 0.7484 (SD=0.1158), n=50


In [8]:
summary = pd.read_csv("outputs/model_comparison_results.csv")
summary = pd.concat([summary, pd.DataFrame([{
    "Model": "XGBoost", "Mean_Macro_F1": np.mean(fold_f1), "Std": np.std(fold_f1)
}])], ignore_index=True).sort_values("Mean_Macro_F1", ascending=False)

print(summary.to_string(index=False))
summary.to_csv("outputs/model_comparison_results.csv", index=False)
print(classification_report(all_true, all_preds, target_names=le.classes_))

             Model  Mean_Macro_F1      Std
           XGBoost       0.748420 0.115831
LogisticRegression       0.743812 0.130790
      RandomForest       0.730598 0.091181
  GradientBoosting       0.723334 0.113563
               MLP       0.683362 0.109444
        RuleEngine       0.667595 0.129009
               SVM       0.652526 0.119232
              precision    recall  f1-score   support

      NORMAL       0.86      0.85      0.85     14980
    STRESSED       0.65      0.67      0.66      6410

    accuracy                           0.79     21390
   macro avg       0.76      0.76      0.76     21390
weighted avg       0.80      0.79      0.80     21390



In [5]:
scores_df = pd.read_csv("outputs/per_fold_f1_scores.csv")
scores_df["XGBoost"] = fold_f1
scores_df.to_csv("outputs/per_fold_f1_scores.csv", index=False)
print("Added. Columns now:", scores_df.columns.tolist())

Added. Columns now: ['LogisticRegression', 'SVM', 'RandomForest', 'GradientBoosting', 'MLP', 'RuleEngine', 'Proposed_Stacking_Ensemble', 'XGBoost']


In [6]:
summary_check = pd.read_csv("outputs/model_comparison_results.csv")
print(summary_check.to_string(index=False))
scores_check = pd.read_csv("outputs/per_fold_f1_scores.csv")
print(scores_check.columns.tolist())

                     Model  Mean_Macro_F1      Std  Accuracy_Mean  Accuracy_SD
Proposed_Stacking_Ensemble       0.772023 0.103286            NaN          NaN
        LogisticRegression       0.743812 0.130790            NaN          NaN
              RandomForest       0.730598 0.091181            NaN          NaN
          GradientBoosting       0.723334 0.113563            NaN          NaN
                       MLP       0.683362 0.109444            NaN          NaN
                RuleEngine       0.667595 0.129009            NaN          NaN
                       SVM       0.652526 0.119232            NaN          NaN
['LogisticRegression', 'SVM', 'RandomForest', 'GradientBoosting', 'MLP', 'RuleEngine', 'Proposed_Stacking_Ensemble', 'XGBoost']


In [7]:
import pandas as pd
import numpy as np

scores_df = pd.read_csv("outputs/per_fold_f1_scores.csv")
xgb_scores = scores_df["XGBoost"]

summary = pd.read_csv("outputs/model_comparison_results.csv")
summary = summary[summary["Model"] != "XGBoost"]  # remove in case a partial row exists
summary = pd.concat([summary, pd.DataFrame([{
    "Model": "XGBoost",
    "Mean_Macro_F1": xgb_scores.mean(),
    "Std": xgb_scores.std()
}])], ignore_index=True).sort_values("Mean_Macro_F1", ascending=False)

summary.to_csv("outputs/model_comparison_results.csv", index=False)
print(summary.to_string(index=False))

                     Model  Mean_Macro_F1      Std  Accuracy_Mean  Accuracy_SD
Proposed_Stacking_Ensemble       0.772023 0.103286            NaN          NaN
                   XGBoost       0.748420 0.117006            NaN          NaN
        LogisticRegression       0.743812 0.130790            NaN          NaN
              RandomForest       0.730598 0.091181            NaN          NaN
          GradientBoosting       0.723334 0.113563            NaN          NaN
                       MLP       0.683362 0.109444            NaN          NaN
                RuleEngine       0.667595 0.129009            NaN          NaN
                       SVM       0.652526 0.119232            NaN          NaN
